# 04 出租车时空出行画像

运营团队想知道本月绿出租车的上车活动如何随地区、日期类型和时间变化。请制作一份时空画像，提出值得进一步核查的运营假设，同时明确样本覆盖的局限。

建议工作量：8—10小时。本工作本是项目起点，默认程序的输出不是完整作业答案。

## 最终成果
- 地区与小时的口径说明
- 时空矩阵及日类型对比
- 总量与日均口径对照
- 运营假设与补充数据清单


## 数据与范围

[NYC TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

遵循 NYC TLC 数据条款，课程仅分发聚合结果

原始56,551行，按上车时间保留当月56,549行。聚合为日期×行政区×小时，不公布单趟轨迹。

- 这里只统计上车记录，不包含未满足需求。
- 绿出租车不能代表全部出租车或全体居民。
- 比较平日与周末时要考虑天数不同。
- 日均每小时与时段总量单位不同。


In [ ]:
from pathlib import Path
import sys, json
candidates = [Path.cwd(), Path.cwd().parent]
ROOT = next((p for p in candidates if (p / "python" / "analyze.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the project-kit folder or its notebooks folder")
sys.path.insert(0, str(ROOT / "python"))
from analyze import run
OUTPUT = ROOT / "outputs" / "student"
OUTPUT.mkdir(parents=True, exist_ok=True)


## 参考分析与口径检查

先运行一次，解释每个指标的分母和单位。打开源码确认数据筛选规则。预测项目这里读取冻结模型的结果，不重新训练。


In [ ]:
project = "taxi"
config = {
  "day": "all",
  "period": "all",
  "normalize": "daily"
}
result = run(project, config, ROOT / "data")
print(json.dumps(result["metrics"], ensure_ascii=False, indent=2))
print("Source SHA-256:", result["sourceSha"])


## 对照实验

以下配置提供一个可运行起点。说明每次只改变了什么，以及还存在哪些混杂条件。增加你自己的对照，不只重复默认结果。


In [ ]:
comparisons = [
  {
    "day": "weekday",
    "normalize": "total"
  },
  {
    "day": "weekend",
    "normalize": "total"
  },
  {
    "day": "weekend",
    "normalize": "daily"
  }
]
experiments = []
for change in comparisons:
    trial = run(project, {**config, **change}, ROOT / "data")
    experiments.append({"project": project, "config": trial["config"], "metrics": trial["metrics"], "sourceSha": trial["sourceSha"]})
    print(json.dumps(experiments[-1], ensure_ascii=False))
(OUTPUT / (project + "-comparison.json")).write_text(json.dumps(experiments, ensure_ascii=False, indent=2), encoding="utf-8")


## 01 样本覆盖

**说明记录代表谁**

解释上车记录与出行需求的区别，说明绿出租车的覆盖限制。

阶段成果：观测单元、覆盖范围和未知地区处理规则。

### 我的证据与解释

在这里填写自己的分析，引用结果行、实验参数或图表。


## 02 时空组织

**形成可解释的矩阵**

查看行政区×24小时矩阵。说明每个单元格的单位，以及没有记录的组如何处理。

阶段成果：矩阵口径和一项时段分布发现。

### 我的证据与解释

在这里填写自己的分析，引用结果行、实验参数或图表。


## 03 口径对照

**比较总量与日均**

切换平日、周末、夜间，比较总量和每日日均。至少保存3次配置，解释排名或差距为何改变。

阶段成果：原始计数、日期分母与归一化后的对照。

### 我的证据与解释

在这里填写自己的分析，引用结果行、实验参数或图表。


## 04 运营假设

**提出可检验的后续问题**

提出2项运营假设并列出所需验证数据。为什么不能仅凭这些记录确定增车数量？

阶段成果：假设、验证方式、可能反例和边界说明。

### 我的证据与解释

在这里填写自己的分析，引用结果行、实验参数或图表。


## 深入分析

- 构建按日期分组的相似日聚类，并验证聚类的稳定性。
- 经许可引入其他交通方式数据，对照时空覆盖，避免直接拼接不同分母。

项目包还包含 extensions.py 的可运行扩展。先安装 python/requirements.txt 中的依赖，再运行下面的命令。


In [ ]:
import subprocess
subprocess.run([sys.executable, str(ROOT / "python" / "extensions.py"), "--project", "taxi", "--output", str(OUTPUT / "extensions")], check=True)


## 提交前自查

- [ ] 矩阵总计与筛选记录数一致。
- [ ] 明确日均的日期分母及夜间定义。
- [ ] 运营建议表述为待验证假设，不推断居民整体需求。

报告应包含研究问题、方法对照、发现、局限、源数据哈希和复现命令。请附代码、配置、结果CSV。阶段文字与实验次数不自动换算成绩。
